<!-- ===== POSITION 1: Title and introduction (insert at the very top) ===== -->

# Naive Divide-and-Conquer Polynomial Multiplication

## Introduction
This notebook multiplies two polynomials using a **divide-and-conquer** strategy
without any optimisation such as Karatsuba's trick.

A polynomial is stored as a NumPy array of coefficients, where the **index is the power of x**:

`[1, 5, 32, 1, 0, 0]`  ≡  `1 + 5x + 32x² + 1x³ + 0x⁴ + 0x⁵`

### The idea
Split each polynomial into a low half and a high half:

- `A(x) = A0(x) + x^k · A1(x)`
- `B(x) = B0(x) + x^k · B1(x)`

Multiplying gives:

`A·B = A0·B0  +  x^k · (A0·B1 + A1·B0)  +  x^(2k) · A1·B1`

This takes **4 recursive multiplications** on half-sized inputs. The partial results are then
shifted by the right power of x and added together. Because it uses 4 sub-products
rather than 3, it is called *naive*.

### Notebook structure
1. Import NumPy
2. `split`, a helper that cuts an array into two halves
3. `multiply`, the recursive multiplication
4. A test case, its results, and a complexity analysis

<!-- ===== POSITION 2: Before the import cell ===== -->

## Step 1: Import Libraries
We only need **NumPy**, which provides:
- fast array slicing (slices are *views*, not copies)
- element-wise arithmetic on arrays (`a0b1 + a1b0`)
- `np.zeros` to allocate the result buffer


In [1]:
import numpy as np

---

<!-- ===== POSITION 4: Before the `split` cell ===== -->

## Step 2: The `split` Helper
**Purpose:** divide a coefficient array into a low half and a high half.

| Item | Description |
|---|---|
| **Input** | `a`, a 1-D NumPy array of coefficients |
| **Output** | `a0`, `a1`, `mid` |

- `mid = a.size // 2` is the split point (floor division).
- `a0 = a[0:mid]` holds the **low-order** coefficients (powers `0 … mid-1`).
- `a1 = a[mid:a.size]` holds the **high-order** coefficients. These are multiplied by `x^mid` in the full polynomial.
- `mid` is returned because it is the **power of x** by which the high half must later be shifted.

**Example:** `[1, 5, 32, 1, 0, 0]` → `a0 = [1, 5, 32]`, `a1 = [1, 0, 0]`, `mid = 3`.
For odd sizes the low half is the smaller one, e.g. size 3 → `a0` has 1 element and `a1` has 2.

---

In [2]:
def split(a):
    mid = a.size//2
    a0 = a[0:mid]
    a1 = a[mid:a.size]
    return a0,a1,mid

<!-- ===== POSITION 6: Before the `multiply` cell ===== -->

## Step 3: The Recursive `multiply` Function
**Purpose:** return the coefficient array of the product of polynomials `a` and `b`.

### Walkthrough
1. **Sizes:** `n = a.size`, `m = b.size`.
2. **Base case:** if either polynomial has a single coefficient, it is just a constant.
   Return `a * b`. NumPy broadcasts the scalar over the other array, which is exactly
   scalar × polynomial.
3. **Divide:** split both inputs.
   `power` is the split point of `a` and `temp` is the split point of `b`.
   Only `power` is used later for shifting.
4. **Conquer:** make four recursive calls: `a0b0`, `a0b1`, `a1b0`, `a1b1`.
5. **Combine:**
   - `middle = a0b1 + a1b0` is the cross term, which shares the same shift.
   - `result = np.zeros(n + m - 1)` is the product of degree `(n-1)` and degree `(m-1)` polynomials,
     which has degree `n+m-2` and therefore `n+m-1` coefficients.
   - Add the pieces into the right positions:

| Piece | Written at index | Meaning |
|---|---|---|
| `a0b0` | `0` | `x⁰` term |
| `middle` | `power` | `x^power` term |
| `a1b1` | `2·power` | `x^(2·power)` term |

6. **Return** the combined array.

### Note on correctness
The shifts assume `a` and `b` split at the **same index** (`power == temp`).
This holds when both arrays have equal length, as in the test below, but not in general
(e.g. sizes 4 and 3 split at 2 and 1). For arbitrary sizes, pad both arrays to the same
power-of-two length, or use `temp` for the `a0b1` shift and `power + temp` for the `a1b1` shift.

In [3]:
def multiply(a,b):
    n = a.size
    m = b.size
    if(n==1 or m==1):
        return a*b
    a0,a1,power = split(a)
    b0,b1,temp  = split(b)


    a0b0 = multiply(a0,b0)
    a0b1 = multiply(a0,b1)
    a1b0 = multiply(a1,b0)
    a1b1 = multiply(a1,b1)

    middle = a0b1+a1b0
    result = np.zeros(n+m-1,dtype =int)
    result[:a0b0.size] += a0b0
    result[power:power+middle.size]+=middle
    result[2*power : 2*power+a1b1.size]+=a1b1

    return result

## Step 4: Test the Implementation
We multiply two degree-5 polynomials:

- `A(x) = 1 + 5x + 32x² + x³`
- `B(x) = 1 + 3x³ + 45x⁴ + 2x⁵`

Both have 6 coefficients, so the product has `6 + 6 − 1 = 11` coefficients (degrees 0 to 10).
The cell prints the resulting coefficient array.

---

<!-- ===== POSITION 10: After the test cell ===== -->

## Results
Output of `print(multiply(a, b))`:

```
[   1    5   32    4   60  323 1453  109    2    0    0]
```

Meaning:
`1 + 5x + 32x² + 4x³ + 60x⁴ + 323x⁵ + 1453x⁶ + 109x⁷ + 2x⁸ + 0x⁹ + 0x¹⁰`

### Verification
Each output coefficient is `c_k = Σ aᵢ · b_(k−i)`. For example:
- `c₃ = a₀b₃ + a₃b₀ = 3 + 1 = 4`
- `c₅ = a₀b₅ + a₁b₄ + a₂b₃ = 2 + 225 + 96 = 323`
- `c₆ = a₁b₅ + a₂b₄ + a₃b₃ = 10 + 1440 + 3 = 1453`

The result has length 11 as predicted, and the last two zeros come from `a₄ = a₅ = 0`.
It matches `np.convolve(a, b)`, which is a quick way to check the function.

### Recursion shape for this input
- Level 0: sizes (6, 6) → split into (3, 3)
- Level 1: sizes (3, 3) → split into 1 and 2 elements
- Level 2: sizes (1, 2), (2, 1), (2, 2) → the (1, ·) cases hit the base case, and (2, 2) splits once more into (1, 1) calls
- Level 3: sizes (1, 1) are multiplied directly

---

In [4]:
a = np.array([1, 5, 32, 1, 0, 0])
b = np.array([1, 0, 0, 3, 45, 2])
print(multiply(a,b))

[   1    5   32    4   60  323 1453  109    2    0    0]


## Time and Space Complexity
Let `n` be the length of each input (assuming `n = m`).

### Time complexity: **O(n²)**
- Each call makes **4** recursive calls on inputs of size `n/2`.
- Splitting is O(1), because NumPy slices are views.
- Combining costs **O(n)**: one `np.zeros`, one array addition and three slice-additions.

Recurrence: `T(n) = 4·T(n/2) + O(n)`

By the Master Theorem (a = 4, b = 2, f(n) = O(n), `n^(log₂4) = n²`), this gives **T(n) = Θ(n²)**.

This is the same as the schoolbook method, so the divide-and-conquer split gives **no speed-up**.
Karatsuba reduces the number of sub-products to 3 and achieves `O(n^1.585)`.
FFT-based multiplication achieves `O(n log n)`.

For unequal lengths `n` and `m` the cost is **O(n·m)**.

### Space complexity: **O(n)** auxiliary
- Recursion depth is `log₂ n`.
- Each frame holds result arrays of size proportional to its input (`n, n/2, n/4, …`), which sum to a geometric series O(n).
- The output array has size `n + m − 1 = O(n)`.
- Slices are views, so they add no extra memory.

Temporary arrays are created and discarded across the whole run, but only O(n) is alive at any moment.
(For unequal lengths, the space is O(n + m).)

| Metric | Naive D&C | Karatsuba | FFT |
|---|---|---|---|
| Time | O(n²) | O(n^1.585) | O(n log n) |
| Space | O(n) | O(n) | O(n) |